# Counting *Tuta absoluta* from RGB trap photographs

Count-guided calibration walkthrough. Run all cells in order from the repository root. **Observed counts select thresholds: same-image error is not independent predictive accuracy.**


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tuta_pipeline import (
    Config, load_image_rgb, compute_connectivity_map,
    segment_connectivity, smooth_binary_mask, find_external_contours,
    filter_contours, load_galleries, classify_retained_contours,
    correct_count, draw_classifications, run_pipeline
)
# Select any filename in examples/metadata.csv.
metadata = pd.read_csv("examples/metadata.csv")
IMAGE_NAME = "photo_01_05_2026_12-01-01cut.png"
IMAGE_PATH = Path("examples") / IMAGE_NAME
GALLERY_DIR = Path("tuta_gallery")
OUTPUT_DIR = Path("outputs/notebook")
OBSERVED_COUNT = int(metadata.set_index("filename").loc[IMAGE_NAME, "observed_count"])
config = Config(min_area=600, max_area=5500, min_solidity=0.43,
                gallery_k=3, min_margin=0.001, filter_mode="optimize",
                optimizer_count_mode="legacy_original")
config.validate()
# Restore interactive display after the module's headless CLI backend setup.
%matplotlib inline


## 1. Load the RGB image

All channel calculations use normalized RGB values between 0 and 1. We preserve the `uint8` RGB image for plotting and colored contour overlays.

In [ ]:
import numpy as np
img_rgb = load_image_rgb(IMAGE_PATH)
img_norm = img_rgb.astype(np.float32) / 255.0
print("Image shape:", img_rgb.shape)
plt.figure(figsize=(9, 6))
plt.imshow(img_rgb); plt.title("Original trap photograph"); plt.axis("off"); plt.show()

## 2. Red-balanced spatial connectivity

The score is the red-balanced spectral feature of the center pixel multiplied by an anisotropic Gaussian average of neighboring features. The image border is `NaN` because a complete neighborhood does not fit there. **This is not graph-shortest-path closeness centrality.**

In [ ]:
closeness_map = compute_connectivity_map(img_norm, config)
plt.figure(figsize=(9, 6))
plt.imshow(closeness_map, cmap="magma")
plt.colorbar(label="Red-balanced spatial connectivity")
plt.title("Connectivity map"); plt.axis("off"); plt.show()

## 3. K-means foreground and morphological cleanup

K-means receives **only finite scores**. The cluster with the larger connectivity centroid is selected as candidate foreground. Morphological closing joins small breaks; opening removes small artifacts.

In [ ]:
binary_mask, cluster_centers = segment_connectivity(closeness_map, config)
smoothed_image = smooth_binary_mask(binary_mask, config)
print("Cluster centers:", cluster_centers)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(binary_mask, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("K-means foreground")
axes[1].imshow(smoothed_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("After closing and opening")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Extract and optimize candidate contours

Observed abundance guides area/solidity threshold selection. The legacy area-based objective differs from the final gallery-corrected count. A baseline within 5% is accepted immediately; otherwise the supplied search runs.


In [ ]:
contours_all = find_external_contours(smoothed_image)
filter_results = filter_contours(
    contours_all, smoothed_image.shape, config, OBSERVED_COUNT
)
kept_contours = filter_results.kept_contours
print("All outer contours:", len(contours_all))
print("Retained after area-solidity:", len(kept_contours))
print("Thresholds used:", filter_results.parameters)
print("Optimizer objective (not final):", filter_results.objective_count)
display(pd.DataFrame(filter_results.history))


## 5. Compare **filtered** contours with the galleries

The gallery must contain `single/` and `touching/` PNG folders; `noise/` is optional. The classifier never reconstructs contours from the colored overlay. Each retained contour produces exactly one output row.

In [ ]:
galleries = load_galleries(GALLERY_DIR)
shape_results = classify_retained_contours(
    kept_contours, filter_results.source_ids, galleries, config
)
print(shape_results["classification"].value_counts(dropna=False))
display(shape_results.head(10))
assert len(shape_results) == len(kept_contours)

## 6. Correct abundance and examine detections

**Your current rule**: count one moth for *every* retained contour and add one more for each gallery-classified touching pair. The gallery labels `noise` and `uncertain` are **diagnostic only**, and still contribute one under this rule. Red = touching, green = single, magenta = noise, orange = uncertain.

In [ ]:
counts = correct_count(kept_contours, shape_results)
print("Retained contours:", counts["n_retained_contours"])
print("Touching pairs:", counts["n_touching"])
print("Final corrected Tuta count:", counts["estimated_tuta"])

overlay_img = draw_classifications(img_rgb, kept_contours, shape_results)
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(img_rgb); axes[0].set_title(f"Original; observed = {OBSERVED_COUNT}")
axes[1].imshow(overlay_img); axes[1].set_title(
    f"Corrected = {counts['estimated_tuta']} "
    f"({counts['n_retained_contours']} contours + {counts['n_touching']} pairs)"
)
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Export intermediate arrays and per-contour classifications

This cell saves the results of the analysis **without rerunning expensive connectivity/K-means processing**. The command-line script `tuta_pipeline.py` additionally writes a full JSON summary and a composite diagnostic figure.

In [ ]:
import cv2
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shape_results.to_csv(OUTPUT_DIR / "contour_classification.csv", index=False)
cv2.imwrite(str(OUTPUT_DIR / "01_kmeans_foreground.png"), binary_mask)
cv2.imwrite(str(OUTPUT_DIR / "02_morphological_mask.png"), smoothed_image)
cv2.imwrite(str(OUTPUT_DIR / "03_size_solidity_mask.png"), filter_results.cleaned_mask)
cv2.imwrite(str(OUTPUT_DIR / "04_classified_overlay.png"),
            cv2.cvtColor(overlay_img, cv2.COLOR_RGB2BGR))
print("Saved to:", OUTPUT_DIR)
import json
from dataclasses import asdict
pd.DataFrame(filter_results.history).to_csv(OUTPUT_DIR / "filter_history.csv", index=False)
(OUTPUT_DIR / "settings.json").write_text(json.dumps(asdict(config), indent=2))
notebook_summary = dict(counts, observed_count=OBSERVED_COUNT, filter_mode=config.filter_mode, optimization_count_mode=config.optimizer_count_mode, optimizer_objective_count_not_final=filter_results.objective_count, final_min_area=filter_results.parameters[0], final_max_area=filter_results.parameters[1], final_min_solidity=filter_results.parameters[2])
(OUTPUT_DIR / "summary.json").write_text(json.dumps(notebook_summary, indent=2))


## Reproducibility and validation

- Keep a frozen copy of the labeled gallery and record thresholds/versions.
- Inspect contours classified as `noise` and `uncertain`: the selected count rule **does not** exclude them.
- A size-solidity filter may reject touching pairs before the gallery sees them.
- Hu moments are only approximately scale-invariant for rasterized silhouettes.
- Evaluate on **held-out trap images**, rather than only those used for selecting thresholds or creating gallery references.
- To run the entire pipeline from a terminal and automatically export `summary.json`, see `README.md`.